In [3]:
# ============================================================
# Notebook 3: Modelling
# EEEM073 - AI and Sustainability
# Models:
#   1. XGBoost baseline (interpretable, fast)
#   2. Custom LSTM with site embeddings (PyTorch, from scratch)
#   3. PINN-LSTM (physics-constrained, sewage dilution dynamics)
# All models built from scratch — no pretrained weights used
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import time
import os
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib

# Set device — use MPS on M4 Mac for GPU acceleration
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Paths
PROCESSED = "../data/processed/"
OUTPUTS = "../outputs/"
os.makedirs(OUTPUTS, exist_ok=True)

# Load data
df = pd.read_csv(f"{PROCESSED}edm_site_imd.csv")
print(f"Dataset loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")

Device: mps
Dataset loaded: (28632, 34)
Columns: ['company', 'site_name', 'permit_ref', 'asset_type', 'grid_ref', 'waterbody_id', 'catchment_name', 'receiving_water', 'total_spill_hours', 'spill_count', 'spill_count_avg', 'edm_years', 'edm_pct_operational', 'unique_id', 'year', 'source_sheet', 'monitor_reliable', 'lat', 'lon', 'company_imd_score', 'spill_severity', 'avg_spill_duration_hrs', 'spill_vs_average', 'high_spiller', 'yoy_change', 'yoy_pct_change', 'reliability_cat', 'coords_available', 'imd_quintile', 'nearest_lad', 'site_imd_score', 'site_imd_decile', 'distance_to_lad_km', 'site_imd_quintile']


In [4]:
# ============================================================
# Feature preparation for modelling
# Target: spill_count (regression)
# Features: engineered variables from EDA
# ============================================================

# --- Select features ---
feature_cols = [
    'spill_count_avg',        # historical average — strongest predictor (r=0.86)
    'edm_pct_operational',    # monitor reliability
    'avg_spill_duration_hrs', # average duration per event
    'spill_severity',         # combined frequency x duration
    'yoy_change',             # year over year trend
    'site_imd_score',         # deprivation score
    'company_imd_score',      # company-level deprivation
    'year',                   # temporal feature
]

target_col = 'spill_count'

# --- Encode company as categorical ---
le_company = LabelEncoder()
df['company_encoded'] = le_company.fit_transform(df['company'].fillna('Unknown'))
feature_cols.append('company_encoded')

# --- Drop rows with nulls in features or target ---
model_df = df[feature_cols + [target_col, 'unique_id', 
                               'site_imd_quintile']].dropna().copy()
print(f"Modelling dataset: {model_df.shape}")
print(f"Target stats:\n{model_df[target_col].describe().round(2)}")

# --- Features and target ---
X = model_df[feature_cols].values
y = model_df[target_col].values

# --- Temporal train/val/test split ---
# Train on 2023, validate and test on 2024
# This prevents data leakage — never split randomly for time series
train_mask = model_df['year'] == 2023
test_mask  = model_df['year'] == 2024

X_train_full = X[train_mask]
y_train_full = y[train_mask]
X_test  = X[test_mask]
y_test  = y[test_mask]

# Split train into train/val (80/20 of 2023 data)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

print(f"\nTrain: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Split strategy: temporal (2023 train/val → 2024 test)")
print(f"Reason: random split would leak future spill patterns into training")

# --- Scale features ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Save scaler and encoders for evaluation notebook
joblib.dump(scaler, f"{PROCESSED}scaler.pkl")
joblib.dump(le_company, f"{PROCESSED}label_encoder_company.pkl")
print("\nScaler saved")

Modelling dataset: (27415, 12)
Target stats:
count    27415.00
mean        32.70
std         38.96
min          0.00
25%          4.00
50%         19.00
75%         49.00
max        366.00
Name: spill_count, dtype: float64

Train: (10564, 9) | Val: (2642, 9) | Test: (14209, 9)
Split strategy: temporal (2023 train/val → 2024 test)
Reason: random split would leak future spill patterns into training

Scaler saved


In [5]:
# ============================================================
# Model 1: XGBoost Baseline
# Why XGBoost: interpretable, handles non-linearity,
# works well with tabular data, compatible with SHAP
# Serves as benchmark for LSTM and PINN-LSTM
# ============================================================

def evaluate_model(y_true, y_pred, model_name):
    """
    Compute regression evaluation metrics.
    Returns dict of metrics for comparison table.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    # Nash-Sutcliffe Efficiency — standard in environmental modelling
    # NSE=1 perfect, NSE=0 same as mean prediction, NSE<0 worse than mean
    nse  = 1 - (np.sum((y_true - y_pred)**2) / 
                np.sum((y_true - np.mean(y_true))**2))
    print(f"\n{model_name} Performance:")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")
    print(f"  NSE  : {nse:.4f}")
    return {'model': model_name, 'rmse': rmse, 'mae': mae, 
            'r2': r2, 'nse': nse}

# --- Hyperparameter tuning — document each choice ---
# n_estimators: 500 with early stopping prevents overfitting
# max_depth: 6 — standard starting point, prevents overly deep trees
# learning_rate: 0.05 — small enough for stable convergence
# subsample: 0.8 — row subsampling reduces overfitting
# colsample_bytree: 0.8 — feature subsampling adds diversity
# min_child_weight: 3 — minimum samples per leaf, reduces noise fitting

xgb_params = {
    'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'random_state': 42,
    'early_stopping_rounds': 30,
    'device': 'cpu'
}

print("Training XGBoost baseline...")
t0 = time.time()

xgb_model = xgb.XGBRegressor(**xgb_params)
xgb_model.fit(
    X_train_s, y_train,
    eval_set=[(X_val_s, y_val)],
    verbose=50
)

xgb_train_time = time.time() - t0
print(f"\nTraining time: {xgb_train_time:.2f}s")
print(f"Best iteration: {xgb_model.best_iteration}")

# Evaluate
xgb_val_pred  = xgb_model.predict(X_val_s)
xgb_test_pred = xgb_model.predict(X_test_s)

xgb_val_metrics  = evaluate_model(y_val,  xgb_val_pred,  "XGBoost (Val)")
xgb_test_metrics = evaluate_model(y_test, xgb_test_pred, "XGBoost (Test)")

# Save model
xgb_model.save_model(f"{PROCESSED}xgb_model.json")
print("\nXGBoost model saved")

# Save training curve
results = xgb_model.evals_result()
plt.figure(figsize=(10, 4))
plt.plot(results['validation_0']['rmse'], color='#378ADD', linewidth=1.5)
plt.xlabel('Boosting round')
plt.ylabel('RMSE')
plt.title('Figure 8: XGBoost validation RMSE during training\n(early stopping prevents overfitting)')
plt.axvline(xgb_model.best_iteration, color='#E24B4A', 
            linestyle='--', label=f'Best iteration: {xgb_model.best_iteration}')
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUTS}fig8_xgb_training.png", dpi=150, bbox_inches='tight')
plt.show()

: 